In [1]:
import torch
from tqdm import trange
import numpy as np
import scipy
import sys
torch.set_grad_enabled(False)
import torch.nn as nn
import math


In [2]:
N = 4
B = 16
H = 64
N1 = int(np.sqrt(N))
print(N1)

u = (torch.randn((B, H, N), dtype=torch.cfloat, device='cpu')).real 
k = (torch.randn((H, N), dtype=torch.cfloat, device='cpu')).real

2


### Real only

In [3]:
def fft_matrix(N):
    n = torch.arange(N)
    k = n.view(-1, 1)
    M = torch.exp(-2j * torch.pi * n * k / N).real
    return M

def compute_twiddle_factors_fft(n, m):
    """Compute the twiddle factors of size n x m"""
    n_a = torch.arange(n).view(-1, 1)
    m_a = torch.arange(m)
    N = n * m
    M = torch.exp(-2j * torch.pi * n_a * m_a / N).real
    return M

def ifft_matrix(N):
    n = torch.arange(N)
    k = n.view(-1, 1)
    M = torch.exp(2j * torch.pi * n * k / N).real
    return M

def compute_twiddle_factors_ifft(n, m):
    """Compute the twiddle factors of size n x m"""
    n_a = torch.arange(n).view(-1, 1)
    m_a = torch.arange(m)
    N = n * m
    M = torch.exp(2j * torch.pi * n_a * m_a / N).real
    return M / N


from torch.nn import functional as F
def ref_fftconv_test(u, k, N):
    L = u.shape[-1]
    u_f = torch.fft.fft(u, n = N).real
    k_f = torch.fft.fft(k, n = N).real
    y_f = u_f * k_f
    y = torch.fft.ifft(y_f, n = N).real[..., :L].to(u.dtype).contiguous()
    return y

def pytorch_test(u, k):
    ############# GET THE INPUTS #############
    f_mat = fft_matrix(N1)
    finv_mat = ifft_matrix(N1)
    
    # Normalization factor to make IFFT exact inverse of FFT
    twiddle_factors_fft = compute_twiddle_factors_fft(N1, N1).to(u.device)
    twiddle_factors_ifft = compute_twiddle_factors_ifft(N1, N1).to(u.device)

    k_f = torch.fft.fft(k, n = N).real

    ############# COMPUTE OUTPUT #############
    # step 1. FFT(U) using FFT matrix
    # compute the FFT
    x = u.reshape(B, H, N1, N1)
    x = x.transpose(-1, -2)
    x = torch.einsum('...i,ij->...j', x, f_mat)
    x = x.transpose(-1, -2)
    x = x * twiddle_factors_fft # (H, sqrt_N, sqrt_N) * (sqrt_N, sqrt_N), pointwise
    x = torch.einsum('...i,ij->...j', x, f_mat)
    # x = x.transpose(-1, -2)

    # pointwise multiplication 
    k_f = k_f.reshape(H, N1, N1).transpose(1,2) # to match the shape of x
    x = x * k_f

    # compute the IFFT
    # x = x.transpose(-1, -2)
    x = x @ finv_mat
    x = x.transpose(-1, -2)
    x = x * twiddle_factors_ifft # (H, sqrt_N, sqrt_N) * (sqrt_N, sqrt_N), pointwise
    x = x @ finv_mat
    x = x.transpose(-1, -2) # necessary to complete the ifft

    x = x.reshape(B, H, N)
    return x

print(f"{u.shape=}, {k.shape=}")
V_real = pytorch_test(u, k)
V_test = ref_fftconv_test(u, k, N) 
print(V_test[0, 0:6, :4])
print(V_real[0, 0:6, :4])
print(torch.allclose(V_real, V_test, atol=1e-3))

u.shape=torch.Size([16, 64, 4]), k.shape=torch.Size([64, 4])
tensor([[-0.4501, -0.5298, -0.4254, -0.5298],
        [ 0.7841,  1.0304,  0.2119,  1.0304],
        [ 0.7418,  0.1688,  0.4411,  0.1688],
        [-0.1613, -0.0828, -0.1343, -0.0828],
        [-0.1060, -1.5650, -2.9689, -1.5650],
        [ 0.1471, -0.3151,  0.2748, -0.3151]])
tensor([[-0.4501, -0.5298, -0.4254, -0.5298],
        [ 0.7841,  1.0304,  0.2119,  1.0304],
        [ 0.7418,  0.1688,  0.4411,  0.1688],
        [-0.1613, -0.0828, -0.1343, -0.0828],
        [-0.1060, -1.5650, -2.9689, -1.5650],
        [ 0.1471, -0.3151,  0.2748, -0.3151]])
True
